# Credit Risk Scoring and Limit Engine - Methodology Walkthrough

This notebook walks through the design decisions and methodology behind the
credit risk scoring and digital lending decisioning system built at Bank Alfalah.

> **Note:** Production data is not included due to banking confidentiality.
> This notebook explains the approach with code excerpts and synthetic examples.

## 1. The Business Problem

The bank needed to launch digital lending products for its existing customers.
The system needed to answer three questions for each of the 891K customers:

1. **Will this customer default?** (probability)
2. **What is their income?** (proxy estimation)
3. **How much credit should we offer?** (limit determination)

These three components form a complete automated credit decisioning pipeline.

## 2. The Class Imbalance Problem

The biggest technical challenge was the extreme class imbalance:
97.5% non-defaulters, only 2.5% defaulters.

Why this matters: a model that always predicts "non-defaulter" gets 97.5%
accuracy but has zero value since it never identifies actual defaults.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Original vs balanced distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Before SMOTE
ax1.bar(['Non-defaulter', 'Defaulter'], [97.5, 2.5], color=['#2196F3', '#E24B4A'])
ax1.set_title('Before KMeans-SMOTE', fontweight='bold')
ax1.set_ylabel('% of customers')
ax1.set_ylim(0, 110)
for i, v in enumerate([97.5, 2.5]):
    ax1.text(i, v + 2, f'{v}%', ha='center', fontweight='bold')

# After SMOTE
ax2.bar(['Non-defaulter', 'Defaulter'], [50, 50], color=['#2196F3', '#E24B4A'])
ax2.set_title('After KMeans-SMOTE', fontweight='bold')
ax2.set_ylabel('% of customers')
ax2.set_ylim(0, 110)
for i, v in enumerate([50, 50]):
    ax2.text(i, v + 2, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Why KMeans-SMOTE instead of regular SMOTE?

Regular SMOTE creates synthetic samples by interpolating between random pairs
of minority class samples. This can create unrealistic samples in sparse regions
of the feature space.

KMeans-SMOTE first clusters the data, then applies SMOTE only within clusters
that contain minority samples. This produces more realistic synthetic data
because the interpolation stays within natural data groupings.

## 3. Feature Selection

We started with 80+ features across 7 data sources. The selection process
used two methods:

**Step 1: Statistical filtering** - Pearson correlation with p-value testing.
Any feature with p-value > 0.05 was eliminated. This removed noisy features
that had no statistically significant relationship with default.

**Step 2: Recursive Feature Elimination (RFE)** - Wrapped around Logistic
Regression. Iteratively removes the least important feature and retrains,
keeping the subset that maximizes model accuracy.

Key findings from feature selection:
- `relationship` (tenure) had the smallest p-value (8.17E-22): longest-tenured
  customers are significantly less likely to default
- Monthly average balances for months 1-4 were eliminated, but month 5 was
  significant: the most recent balance matters more than historical ones
- Transaction frequency (weekly POS, weekly UBP) was more predictive than
  transaction amounts

## 4. Model Architecture

The production model is a Multi-Layer Perceptron with:
- Input layer: 35 features (after encoding, ~65 neurons due to one-hot)
- Hidden layer 1: 25 neurons
- Hidden layer 2: 5 neurons
- Output layer: 2 neurons (defaulter/non-defaulter probability)
- Optimizer: Adam
- Regularization: alpha = 1e-5

In [ ]:
# The actual model definition from our code
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    solver='adam',
    alpha=1e-5,
    hidden_layer_sizes=(25, 5),
    random_state=125
)

print(f"Architecture: Input -> {25} -> {5} -> Output")
print(f"Solver: {mlp.solver}")
print(f"Regularization (alpha): {mlp.alpha}")

## 5. The Limit Determination Engine

The ML model outputs a probability. The business needs a credit limit.
The conversion happens through a three-step process:

1. **Probability to risk band**: 0-0.6 = Low-Medium, 0.6-0.8 = High, 0.8-1.0 = Very High
2. **Income to salary band**: Based on proxy income (6-month avg deposit balance)
3. **Multiplier lookup**: Cross-reference risk band x salary band
4. **Limit = proxy_income x multiplier**

This design separates the ML model (which predicts risk) from the business
policy (which determines how much risk to accept at each income level).
The multiplier matrix can be updated by the business team without retraining
the model.

## 6. Production Pipeline

The production scoring pipeline uses serialized artifacts to ensure consistency:

```python
# Training phase: serialize everything
joblib.dump(mlp, 'finalized_model.sav')
joblib.dump(min_max_scaler, 'MinMaxScaler.sav')
joblib.dump(lb_category, 'LabelEncoder.sav')
joblib.dump(SelectedColumns, 'SelectedColumns.sav')

# Scoring phase: load and apply
loaded_model = joblib.load('finalized_model.sav')
probabilities = loaded_model.predict_proba(eval_set)[:, 1]
```

This separation means the scoring pipeline can run independently,
on a schedule, without retraining the model each time.

## 7. What I Would Do Differently Today

- **Use temporal validation** instead of random train/test split to simulate
  real deployment conditions
- **Try gradient boosting with proper tuning** - the XGBoost benchmark had
  minimal hyperparameter optimization and might have outperformed MLP
- **Add loss-given-default modeling** - not all defaults are equal; a customer
  who misses one payment and recovers is very different from a full write-off
- **Address reject inference bias** - the model only sees customers who were
  given credit cards, creating selection bias
- **Calibrate the multiplier matrix** using historical loss data rather than
  business judgment alone